# Get to Know a Dataset: UCSF-ProstateMR

This notebook serves as a guided tour of the [UCSF-ProstateMR](https://registry.opendata.aws/ucsf-prostatemr) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).


### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The `ucsf-prostatemr` S3 bucket has a flat structure with two types of files at the top level:

1. **`scanner_info.csv`** — a metadata table listing the MRI scanner manufacturer and model for each exam (anonymized MRN, accession number, GE scanner model)
2. **973 HDF5 files** (0.8–9.2 GB each, ~1.8 GB average) — one file per patient exam, named by anonymized accession number (e.g., `10040446.hdf5`)

There are no subdirectory prefixes. Each HDF5 file is self-contained: it stores the registered MRI image volumes, segmentation masks, and all clinical annotations for that exam as internal datasets and attributes.

Full documentation: https://github.com/LarsonLab/UCSF-ProstateMR




First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (install using pip or conda):
#
# boto3 >= 1.38
# h5py >= 3.10
# numpy >= 1.24
# matplotlib >= 3.7
# pandas >= 2.0
# scipy >= 1.10

import boto3
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from botocore import UNSIGNED
from botocore.config import Config


Next, we define the S3 bucket location and create a public (unsigned) boto3 client to list the dataset contents.


In [ ]:
# Location of the S3 bucket for this dataset
BUCKET = 'ucsf-prostatemr'

# Create a boto3 S3 client. Because this is a public bucket, we don't need to sign requests.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# List all objects in the bucket
response = s3.list_objects_v2(Bucket=BUCKET)
objects = response['Contents']

# Separate HDF5 exam files from metadata CSV files
hdf5_files = [o['Key'] for o in objects if o['Key'].endswith('.hdf5')]
csv_files  = [o['Key'] for o in objects if o['Key'].endswith('.csv')]

print(f"Total objects:  {len(objects)}")
print(f"HDF5 files:     {len(hdf5_files)}")
print(f"CSV files:      {len(csv_files)}")
print(f"\nCSV files:")
for f in csv_files:
    size = next(o['Size'] for o in objects if o['Key'] == f)
    print(f"  {f}  ({size / 1024:.1f} KB)")
print(f"\nExample HDF5 filenames:")
for key in hdf5_files[:5]:
    size = next(o['Size'] for o in objects if o['Key'] == key)
    print(f"  {key}  ({size / 1e9:.2f} GB)")


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats?

All 973 patient exams are stored as **HDF5** (Hierarchical Data Format version 5) files. HDF5 is a binary format designed for large scientific datasets — it supports n-dimensional arrays, hierarchical grouping, and per-dataset metadata (attributes), all within a single portable file.

Each `.hdf5` file contains:

| Dataset key | Shape | Description |
|---|---|---|
| `T2` | 512×512×D | T2-weighted MRI volume |
| `ADC_regis` | 512×512×D | ADC map, registered to T2 |
| `DWI_regis` | 512×512×D | High b-value DWI, registered to T2 |
| `T2_ADC_DWI_resample` | 3×H×W×D | All three sequences resampled to isotropic voxels |
| `T2_ADC_DWI_equ` | 3×H×W×D | Histogram-equalized version of the above |
| `SSeg_predicted_wg_proc` | H×W×D | Whole-gland prostate segmentation mask |
| `Lesion1_mask` | H×W×D | PI-RADS lesion segmentation mask |
| `sextant_masks` | 6×H×W×D | Prostate sextant region masks |

Per-case clinical attributes stored on the file root include: `Age`, `PSA_density_ng/ml2`, `Prostate_Vol_cc`, `MRI1_ISUP`, `Lesion_PIRADS`, `Lesion1_Zone`, `Patient_Race`, and `Coil_Anno`.

HDF5 files are read in Python using the `h5py` library.


### Q: Can you show us an example of downloading and loading data from your dataset?

We download a single HDF5 file from S3 and inspect its internal structure using `h5py`.


In [ ]:
import tempfile, os

# Select the first HDF5 file as our example
example_key = hdf5_files[0]

# Download the file to a local temporary directory
local_path = os.path.join(tempfile.gettempdir(), example_key)
print(f"Downloading {example_key} ...")
s3.download_file(BUCKET, example_key, local_path)
print(f"Saved to {local_path}")


In [ ]:
# Open the HDF5 file in read-only mode
with h5py.File(local_path, 'r') as f:

    # Each top-level key is a dataset (image volume, mask, or label array)
    print("=== Image datasets ===")
    for key in f.keys():
        ds = f[key]
        try:
            dtype_str = str(ds.dtype)
        except ValueError:
            dtype_str = 'unknown'
        print(f"  {key:<40} shape={ds.shape}, dtype={dtype_str}")

    # Per-case clinical metadata is stored as HDF5 attributes on the file root
    print("\n=== Clinical attributes ===")
    for k, v in f.attrs.items():
        print(f"  {k}: {v}")


We can inspect the internal structure of an HDF5 file to see all available datasets and clinical attributes.
The datasets include the registered MRI image volumes, segmentation masks, and derived label arrays.
The clinical attributes store per-case metadata such as patient age, PSA density, biopsy ISUP grade, and lesion zone.


In [ ]:
def load_exam(hdf5_path):
    with h5py.File(hdf5_path, 'r') as f:
        # Use native-resolution sequences (512×512×24) to match the mask dimensions
        T2  = f['T2'][:]
        ADC = f['ADC_regis'][:]
        DWI = f['DWI_regis'][:]
        imgs = np.stack([T2, ADC, DWI], axis=0)   # 3 x 512 x 512 x 24

        # Segmentation masks — all at native T2 resolution (512×512×24)
        wg_mask  = f['SSeg_predicted_wg_proc'][:]
        lesion   = f['Lesion1_mask'][:]
        sextants = f['sextant_masks'][:]
        attrs    = dict(f.attrs)
    return imgs, wg_mask, lesion, sextants, attrs

imgs, wg_mask, lesion, sextants, attrs = load_exam(local_path)

# Print array shapes to confirm the data loaded correctly
print(f"Image array shape (C x H x W x D): {imgs.shape}")
print(f"Whole-gland mask shape:             {wg_mask.shape}")
print(f"Lesion mask shape:                  {lesion.shape}")
print(f"Sextant masks shape:                {sextants.shape}")

# Print key clinical attributes for this exam
print(f"\nAge:         {attrs['Age']:.0f}")
print(f"PSA density: {attrs['PSA_density_ng/ml2']:.3f} ng/ml²")
print(f"ISUP grade:  {int(attrs['MRI1_ISUP'])}")
print(f"Lesion zone: {attrs['Lesion1_Zone']}")

### Q: Show us a visual from your dataset that illustrates something informative or exciting.

We display the three MRI sequences (T2, ADC, DWI) for the most informative axial slice, with the whole-gland contour in cyan and the lesion mask in red. A second figure shows the prostate sextant regions overlaid on T2.


In [ ]:
def plot_exam(imgs, wg_mask, lesion, sextants, attrs):
    # Select the axial slice with the largest lesion area; fall back to middle slice if no lesion
    lesion_sum = lesion.sum(axis=(0, 1))
    slice_idx = lesion_sum.argmax() if lesion_sum.max() > 0 else imgs.shape[-1] // 2

    # Extract the 2D slice for each sequence and the corresponding masks
    T2  = imgs[0, :, :, slice_idx]
    ADC = imgs[1, :, :, slice_idx]
    DWI = imgs[2, :, :, slice_idx]
    wg  = wg_mask[:, :, slice_idx]
    les = lesion[:, :, slice_idx]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, img, title in zip(axes, [T2, ADC, DWI], ['T2', 'ADC', 'DWI']):
        ax.imshow(img, cmap='gray', origin='lower')
        # Whole-gland contour in cyan
        ax.contour(wg, levels=[0.5], colors='cyan', linewidths=1)
        # Lesion contour in red (only if a lesion is present in this slice)
        if les.sum() > 0:
            ax.contour(les, levels=[0.5], colors='red', linewidths=1.5)
        ax.set_title(title, fontsize=14)
        ax.axis('off')

    # Annotate the figure with key clinical attributes from the HDF5 file
    isup = int(attrs.get('MRI1_ISUP', 0))
    pirads = attrs.get('Lesion_PIRADS', [float('nan')])[0]
    fig.suptitle(
        f"Age: {attrs['Age']:.0f}  |  PSA density: {attrs['PSA_density_ng/ml2']:.3f} ng/ml²"
        f"  |  ISUP: {isup}  |  PI-RADS: {pirads:.0f}  |  Slice: {slice_idx}",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

plot_exam(imgs, wg_mask, lesion, sextants, attrs)


In [ ]:
import io

# Load scanner metadata CSV directly from S3 into a pandas DataFrame
csv_obj = s3.get_object(Bucket=BUCKET, Key='scanner_info.csv')
scanner_df = pd.read_csv(io.BytesIO(csv_obj['Body'].read()))
print(f"Scanner models in dataset:\n{scanner_df['Model'].value_counts().to_string()}\n")

# Collect clinical attributes from all HDF5 files in the bucket
# Files are downloaded one at a time to a temporary directory to minimize disk usage
records = []
for key in hdf5_files:
    tmp = os.path.join(tempfile.gettempdir(), key)
    s3.download_file(BUCKET, key, tmp)
    with h5py.File(tmp, 'r') as f:
        records.append({
            'ISUP':         f.attrs.get('MRI1_ISUP', float('nan')),
            'Age':          f.attrs.get('Age', float('nan')),
            'PSA_density':  f.attrs.get('PSA_density_ng/ml2', float('nan')),
            'Prostate_Vol': f.attrs.get('Prostate_Vol_cc', float('nan')),
        })
    os.remove(tmp)  # Delete after reading to free disk space

df = pd.DataFrame(records)

# Plot three clinical distributions across the full dataset
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Biopsy ISUP grade group (0 = benign, 1–5 = increasing cancer severity)
df['ISUP'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_xlabel('ISUP Grade Group')
axes[0].set_ylabel('Count')
axes[0].set_title('Biopsy Grade Distribution')
axes[0].tick_params(axis='x', rotation=0)

# Patient age at time of MRI exam
axes[1].hist(df['Age'].dropna(), bins=20, color='salmon', edgecolor='k')
axes[1].set_xlabel('Age (years)')
axes[1].set_title('Patient Age Distribution')

# PSA density — PSA level normalized by prostate volume
axes[2].hist(df['PSA_density'].dropna(), bins=20, color='mediumseagreen', edgecolor='k')
axes[2].set_xlabel('PSA Density (ng/ml²)')
axes[2].set_title('PSA Density Distribution')

plt.tight_layout()
plt.show()
print(df.describe().round(2))


### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?


**How can we use diverse datatypes and sources available to improve prostate cancer classification from MRI?** 

We showed that a mixed supervision approach as well as using diverse datasets can be used to leverage all available data and significantly improve detection of clinically-significant prostate cancer from MRI.

Pathology results, typically Gleason grading, provide one of the best sources of ground-truth for determining clinically-significant prostate cancer.  These grades come with varying levels of spatial information, most commonly only the sextant region of a biopsy is known, but also they maybe based on a MRI-guided biopsy or in rare cases cover the whole prostate when based on surgical specimens.
We found that using mixed supervision that appropriately weights these different types of grading data and their spatial localization information led to significant improvements in a deep learning model of prostate cancer detection and classification.

We also experimented with pre-processing and federated learning to train on MRI data from different institutions that had very different imaging characteristics, showing that this can lead to better generalization.  This dataset has also been used in the [PI-CAI project](https://pi-cai.grand-challenge.org) which aims to create very general models for prostate cancer detection and classification from MRI.

```
Rajagopal A, Westphalen AC, Velarde N, Simko JP, Nguyen H, Hope TA, Larson PE, Magudia K. 
Mixed supervision of histopathology improves prostate cancer classification from MRI. 
IEEE transactions on medical imaging. 2024 Mar 28;43(7):2610-22. doi: 10.1109/TMI.2024.3382909
https://pmc.ncbi.nlm.nih.gov/articles/PMC11361281/
```

```
Rajagopal A, Redekop E, Kemisetti A, Kulkarni R, Raman S, Sarma K, Magudia K, Arnold CW, Larson PE. 
Federated learning with research prototypes: application to multi-center MRI-based detection of prostate cancer with diverse histopathology. 
Academic radiology. 2023 Apr 1;30(4):644-57. doi: 10.1016/j.acra.2023.02.012
https://pmc.ncbi.nlm.nih.gov/articles/PMC10869141/
```

**Does PSA density differ between benign and clinically significant prostate cancer?**

PSA density (PSA level divided by prostate volume) is a clinical marker used to improve specificity of prostate cancer detection. We test whether it is significantly higher in cases with clinically significant cancer (ISUP ≥ 2) versus benign findings (ISUP 0).


In [ ]:
benign    = df[df['ISUP'] == 0]['PSA_density'].dropna()
cs_cancer = df[df['ISUP'] >= 2]['PSA_density'].dropna()

stat, pval = stats.mannwhitneyu(benign, cs_cancer, alternative='less')

print(f"Benign (ISUP 0)       — median PSA density: {benign.median():.3f} ng/ml²  (n={len(benign)})")
print(f"CS Cancer (ISUP ≥ 2)  — median PSA density: {cs_cancer.median():.3f} ng/ml²  (n={len(cs_cancer)})")
print(f"Mann-Whitney U p-value: {pval:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([benign, cs_cancer], tick_labels=['Benign\n(ISUP 0)', 'CS Cancer\n(ISUP ≥ 2)'], widths=0.4)
ax.set_ylabel('PSA Density (ng/ml²)')
ax.set_title('PSA Density vs. Biopsy Grade')
plt.tight_layout()
plt.show()


### Q: What is one unanswered question that you think could be answered using these data?


**Can prostate MRI results be used to confidently recommend that patients NOT proceed to biopsy?**

It is very likely that many prostate biopsies can be avoided, especially since the majority of prostate cancers are benign or low-grade, meaning not clinically-significant, and will not lead to any adverse outcomes.  The challenge is to determine, with confidence, which patients have such a low likelihood of having clinically-significant disease that they can avoid biopsy, as this procedure comes with significant costs as well as some potential for adverse outcomes.

This dataset has a robust set of pathology results, both sextant and guided biopsies, provided with the multiparametric MRI data.  We have confidence that the majority of clinically-significant disease in these patients has been detected.  This could be used to guide algorithm development to answer this question.  Through our collaboration with [PI-CAI project](https://pi-cai.grand-challenge.org), we also know that this dataset can be meaningfully combined with data from other institutions, which will help to address this and other open questions.

**Challenge: Predict clinically significant prostate cancer from multi-parametric MRI using mixed supervision.**

The dataset provides two levels of spatial annotation for each exam: (1) PI-RADS lesion masks from MRI-targeted biopsies (precise, strong supervision) and (2) sextant-level biopsy grades from systematic biopsies (coarser, weak supervision). A model that learns from both simultaneously — assigning lesion-level predictions where masks are available and sextant-level predictions elsewhere — could generalize better than models trained on either alone.

**Suggested starting point:**
- Input: `T2_ADC_DWI_equ` cropped to the whole-gland mask (`SSeg_predicted_wg_proc`)
- Labels: `MRI1_ISUP` for case-level classification, `Lesion1_mask` for lesion localization
- Baseline: 3D CNN or U-Net trained on the case-level label, evaluated by AUC for ISUP ≥ 2

**Known challenges:**
- Scanner heterogeneity (four GE scanner models) requires normalization or harmonization
- Class imbalance: benign cases outnumber high-grade cancer
- Multi-site generalization — see [Rajagopal et al. 2023](https://doi.org/10.1016/j.acra.2023.02.012) for federated learning approaches on this dataset
